# STAC AI Workshop · NYC · 21 October 2026

## 05 · Did the signal predict the future—or read it?

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Compare the same opportunities under valid and leaked information
2. Recompute returns after explicit costs
3. Enforce availability before a simulated order
4. Distinguish a constructed profit from market alpha

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/05_alpha_risk.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/05_alpha_risk.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

In [ ]:
# COLAB_BOOTSTRAP · download the public workshop helpers and saved fixtures.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    import os, subprocess
    workshop_root = Path("/content/STAC_AI_Bootcamp")
    if not (workshop_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mfrdixon/STAC_AI_Bootcamp.git", str(workshop_root)], check=True)
    os.chdir(workshop_root / "notebooks")
    sys.path.insert(0, str(workshop_root))
    sys.path.insert(0, str(workshop_root / "notebooks"))
    print("Workshop helpers and saved fixtures ready. No model API call was made.")

<!-- WORKSHOP-CODE-GUIDE C01 -->
### Code cell C01 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C01:4** | `notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C01 · code cell
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

## Class Exercise · twelve independent opportunities
This matches **Web Section 5**. Each opportunity receives $10,000, independently; profits are summed, not compounded. BUY is held for five sessions. CASH means no trade and earns zero. Fees apply twice to initial notional.

The future-derived signal deliberately reads the realized return. This construction demonstrates leakage; it is not an investment strategy or a measured LLM output.

<!-- WORKSHOP-CODE-GUIDE C02 -->
### Code cell C02 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C02:2** | `data=load_fixture("section5-replay.json")` | Load a saved teaching or recorded-result fixture; this is not a live model call. |
| **C02:3** | `results={"Available at the time":financial_replay(data,"valid",True),"Future-derived / gate OFF":financial_replay(data,"leaked",False),"Future-derived / gate ON":financial_replay(data,"leaked",True)}` | Run the declared simulated trading rule and costs on the fixed opportunity ledger. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C02 · code cell
data=load_fixture("section5-replay.json")
results={"Available at the time":financial_replay(data,"valid",True),"Future-derived / gate OFF":financial_replay(data,"leaked",False),"Future-derived / gate ON":financial_replay(data,"leaked",True)}
show_table([{"Setup":k,"Trades":v["trades"],"Rejected signals":v["blocked"],"Net P&L ($)":round(v["net_pnl"],2),"Net bp/opportunity":round(v["net_bps"],2),"Worst opportunity ($)":round(v["worst_pnl"],2)} for k,v in results.items()])
assert results["Future-derived / gate ON"]["trades"]==0
assert results["Future-derived / gate ON"]["blocked"]==len(data["observations"])
print("No trade (cash) benchmark: $0. No interest, annualization or compounding.")

## Inspect the ledger
The observed return is used for scoring only. The invalid future-derived signal uses it to choose the action, creating look-ahead leakage. The gate uses availability metadata, not future returns.

<!-- WORKSHOP-CODE-GUIDE C03 -->
### Code cell C03 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C03:2** | `replay=results["Future-derived / gate OFF"]` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C03 · code cell
replay=results["Future-derived / gate OFF"]
show_table([{"Opportunity":r["id"],"Decision":r["decision_time"][:10],"Signal available":r["available"][:10],"Signal":r["signal"],"Time eligible":r["eligible"],"Realized return (%)":round(100*r["realized_return"],2),"Net P&L ($)":round(r["pnl"],2)} for r in replay["rows"]])

## Change trading costs
For a BUY, $r_{net}=r_{realized}-2f/10{,}000$, where f is the fee in basis points **each way**. This simple convention applies both fees to initial notional; it is not the earlier four-way experiment's exact exit-notional fee convention. Keep conventions explicit when comparing results.

<!-- WORKSHOP-CODE-GUIDE C04 -->
### Code cell C04 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C04:2** | `show_table([{"Fee each way (bp)":fee,"Time-valid net P&L ($)":round(financial_replay(data,"valid",True,fee)["net_pnl"],2),"Leaked / unguarded P&L ($)":round(financial_replay(data,"leaked",False,fee)["net_pnl"],2)} for fee in (0,15,30)])` | Run the declared simulated trading rule and costs on the fixed opportunity ledger. |
| **C04:3** | `references=load_fixture("section567-reference.json")["financial"]` | Load a saved teaching or recorded-result fixture; this is not a live model call. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C04 · code cell
show_table([{"Fee each way (bp)":fee,"Time-valid net P&L ($)":round(financial_replay(data,"valid",True,fee)["net_pnl"],2),"Leaked / unguarded P&L ($)":round(financial_replay(data,"leaked",False,fee)["net_pnl"],2)} for fee in (0,15,30)])
references=load_fixture("section567-reference.json")["financial"]
for ref in references:
    got=financial_replay(data,ref["mode"],ref["gate"],ref["fee"])
    for key in ("trades","blocked","net_pnl","net_bps","worst_pnl"):
        assert math.isclose(got[key],ref["result"][key],abs_tol=1e-10)
    for a,b in zip(got["rows"],ref["result"]["rows"]):
        for key in ("signal","available","eligible","blocked","trade"): assert a[key]==b[key]
        assert math.isclose(a["net"],b["net"],abs_tol=1e-12)
print("PASS: all 12 mode/gate/fee combinations match the browser.")

## Learning check
Does blocking future information guarantee a good strategy? **No.** It protects the information boundary. Prediction, realistic costs and robustness need separate evidence.

Does the lowest individual P&L estimate VaR? **No.** It is a descriptive worst opportunity in this small fixture.

A point-in-time archive is required to establish real availability; this exercise only checks supplied timestamps.

## Main takeaway
Freeze the information boundary before interpreting returns. An attractive backtest can be explained by leakage rather than prediction; a gate can remove that invalid advantage without creating alpha.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.

## Answer checks · default configuration

- Default fixed ledger: **12 independent opportunities**, each with its declared notional and costs; no compounding or annualization.
- Available-at-the-time signal: **7 trades, 0 blocked**, net P&L **−$50**, approximately **−4.17 bp per opportunity** at the default fee.
- Future-derived signal, gate OFF: **7 trades**, net P&L **+$870**, **+72.5 bp per opportunity**. It is invalid predictive evidence because it uses later information.
- Future-derived signal, gate ON: **0 trades, 12 blocked, $0 P&L**. Blocking leakage is enforcement, not a profitable forecasting model.
- No-trade cash baseline is **$0**, with interest deliberately excluded. Increasing fees reduces the return of executed buys under the specified linear round-trip cost.
- The reference assertions compare all **12 mode/gate/fee combinations** with the recorded browser calculation. They do not validate real market alpha.

If your result differs, check which input, selected case, data version or optional switch changed. For a live model, inspect actual tool calls and returned evidence; do not assume the recorded result is guaranteed.